# 실전 13-3: 심화 에이전틱 사고 (Reflection & Tree of Thoughts)

## 1. Reflection / Reflexion 이란?
- 인간이 글을 쓸 때 초고를 쓴 뒤, 다시 읽어보고 퇴고를 거쳐 최종본을 내는 것과 똑같은 원리입니다.
- **Generator(생성자)** 노드가 답변을 만들면, **Critic(비평가)** 노드가 "이 답변의 틀린 점, 누락된 점"을 지적(비판)합니다. 그 지적을 바탕으로 Generator가 다시 답변을 수정하는 과정을 반복합니다.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

## 2. Reflection 파이프라인 구현
단순 체인으로 초고 작성 -> 비평 -> 수정본 작성 흐름을 만들어봅니다.

In [2]:
topic = "현대 직장인의 번아웃 증후군 원인과 해결책"

# 1. 초고 작성 (Generator)
draft_prompt = ChatPromptTemplate.from_template("다음 주제에 대해 짧은 블로그 글 초고를 작성해줘: {topic}")
draft_chain = draft_prompt | llm

draft_content = draft_chain.invoke({"topic": topic}).content
print("=== [1. 초고 작성 완료] ===")
print(draft_content[:200] + "... (중략)")

# 2. 비평가 (Critic)
critic_prompt = ChatPromptTemplate.from_template("당신은 엄격한 편집장입니다. 다음 글을 읽고 부족한 점, 논리적 비약, 추가해야 할 사항을 신랄하게 비판(Critic)하세요.\n\n[글]:\n{draft}")
critic_chain = critic_prompt | llm

critic_feedback = critic_chain.invoke({"draft": draft_content}).content
print("\n=== [2. 편집장의 비판(Reflection) 완료] ===")
print(critic_feedback)

# 3. 수정본 작성 (Revised Generator)
revise_prompt = ChatPromptTemplate.from_template("당신은 작가입니다. 이전 작성한 초고와 편집장의 비판을 꼼꼼히 반영하여 더 완벽한 최종본을 작성하세요.\n\n[초고]:\n{draft}\n\n[편집장 비판]:\n{feedback}")
revise_chain = revise_prompt | llm

final_content = revise_chain.invoke({"draft": draft_content, "feedback": critic_feedback}).content
print("\n=== [3. 최종 수정본 작성 완료] ===")
print(final_content[:300] + "... (중략)")

=== [1. 초고 작성 완료] ===
### 현대 직장인의 번아웃 증후군: 원인과 해결책

현대 사회에서 직장인들은 끊임없는 업무 압박과 경쟁 속에서 살아가고 있습니다. 이러한 환경은 종종 '번아웃 증후군'으로 이어지며, 이는 신체적, 정서적, 정신적 탈진 상태를 의미합니다. 번아웃 증후군은 단순한 피로감을 넘어서, 직무에 대한 무관심과 비효율성을 초래할 수 있어 개인과 조직 모두에게 심각한 ... (중략)

=== [2. 편집장의 비판(Reflection) 완료] ===
이 글은 현대 직장인의 번아웃 증후군에 대한 원인과 해결책을 제시하고 있지만, 몇 가지 부족한 점과 논리적 비약이 존재합니다. 다음은 그에 대한 비판입니다.

### 1. 원인 설명의 깊이 부족
각 원인에 대한 설명이 다소 피상적입니다. 예를 들어, "과중한 업무"라는 원인은 구체적인 사례나 통계 없이 단순히 언급되고 있습니다. 독자가 이 원인의 심각성을 이해할 수 있도록 실제 사례나 연구 결과를 인용하는 것이 필요합니다. 또한, "일과 삶의 불균형"에 대한 설명도 개인의 심리적, 사회적 요인에 대한 논의가 부족합니다. 이 문제는 단순히 시간 관리의 문제로 축소될 수 없으며, 사회적 구조와 문화적 요인도 고려해야 합니다.

### 2. 해결책의 실효성
제시된 해결책들은 일반적이고 추상적입니다. "업무 재조정"이나 "소통 강화"와 같은 해결책은 구체적인 실행 방안이 부족하여 실질적인 도움이 되기 어렵습니다. 예를 들어, "업무량을 조절하고 우선순위를 정하는 것"이 어떻게 이루어져야 하는지에 대한 구체적인 방법론이 필요합니다. 또한, "목표 설정"에 대한 부분도 단순히 목표를 세분화하는 것만으로는 부족하며, 목표 달성을 위한 구체적인 전략이나 도구에 대한 언급이 필요합니다.

### 3. 논리적 비약
글의 마지막 부분에서 "건강한 직장 문화를 만들어 나가는 것이 모든 직장인의 행복과 생산성을 높이는 길"이라고 결론짓고 있지만, 이 주장을 뒷받침할 만한 논리적 근거가 부족합니다. 건강한 직장 문

## 3. Tree of Thoughts (ToT) 개념 소개
- 2023년 프린스턴 대학교와 구글 딥마인드가 제안한 기법입니다.
- 보통 LLM은 첫 단어를 뱉고 나면 그 길로 쭉 밀고 나갑니다 (Chain of Thought).
- 하지만 ToT는 **"여러 개의 아이디어(가지)를 동시에 내놓고 -> 비평가가 각 아이디어의 점수를 매겨 -> 점수가 높은 가지만 살려서 다음 단계로 진행"** 하는 방식입니다. (마치 체스 알파고의 탐색 트리와 비슷합니다.)

In [3]:
# ToT의 흐름을 흉내내는 컨셉 코드입니다.

problem = "사막 한가운데 조난당했을 때 살아남기 위한 가장 중요한 초기 행동은?"

# 1. 3가지 다른 해결책(가지) 생성
generate_thoughts_prompt = ChatPromptTemplate.from_template("다음 상황에서 취할 수 있는 '완전히 다른 3가지 전략'을 제안해봐.\n상황: {problem}")
thoughts = (generate_thoughts_prompt | llm).invoke({"problem": problem}).content
print("=== [1. 3가지 사고의 가지(Tree) 생성] ===")
print(thoughts)

# 2. 각 전략 평가 및 점수 매기기 (비평가)
evaluate_thoughts_prompt = ChatPromptTemplate.from_template("다음 3가지 전략을 분석하고, 생존 확률 측면에서 어떤 전략이 가장 우수한지 1~10점 사이로 평가해서 가장 높은 점수를 받은 전략 하나만 선택해.\n\n[전략들]:\n{thoughts}")
evaluation = (evaluate_thoughts_prompt | llm).invoke({"thoughts": thoughts}).content
print("\n=== [2. 가장 우수한 전략 가지치기(선택)] ===")
print(evaluation)

=== [1. 3가지 사고의 가지(Tree) 생성] ===
사막 한가운데 조난당했을 때 살아남기 위한 초기 행동으로 고려할 수 있는 완전히 다른 3가지 전략은 다음과 같습니다.

1. **물 자원 확보 및 보존 전략**:
   - 조난 상황에서 가장 중요한 자원 중 하나는 물입니다. 주변 환경을 탐색하여 물이 있을 가능성이 있는 장소(예: 식물 근처, 낮은 지대 등)를 찾아보세요. 만약 물을 찾았다면, 가능한 한 아껴서 마시고, 체온을 조절하기 위해 물을 적신 천으로 몸을 감싸는 것도 도움이 됩니다.

2. **신호 보내기 전략**:
   - 구조 요청을 위해 신호를 보내는 것이 중요합니다. 주변에 있는 자원을 활용하여 큰 불을 피우거나, 반사 가능한 물체(예: 거울, 금속 등)를 사용해 햇빛을 반사시켜 구조대에 신호를 보낼 수 있습니다. 또한, 큰 글씨로 "SOS"나 "HELP"와 같은 신호를 모래에 쓰는 것도 효과적입니다.

3. **대피소 구축 전략**:
   - 사막의 극심한 온도 변화에 대비하기 위해 대피소를 만드는 것이 중요합니다. 주변의 돌이나 나뭇가지를 이용해 그늘을 만들거나, 모래를 파서 작은 구멍을 만들어 더위를 피할 수 있는 공간을 확보하세요. 이렇게 하면 낮의 뜨거운 햇볕을 피하고, 밤에는 체온을 유지하는 데 도움이 됩니다.

이 세 가지 전략은 조난 상황에서 생존 확률을 높이는 데 중요한 역할을 할 수 있습니다.

=== [2. 가장 우수한 전략 가지치기(선택)] ===
각 전략의 생존 확률 측면에서의 효과를 분석해보겠습니다.

1. **물 자원 확보 및 보존 전략** (점수: 10점)
   - 물은 생존에 가장 중요한 요소 중 하나입니다. 사막에서는 탈수 위험이 크기 때문에 물을 확보하고 아껴서 사용하는 것이 생존 확률을 극대화합니다. 체온 조절을 위한 물의 활용도 매우 중요합니다. 이 전략은 생존에 필수적인 요소를 직접적으로 다루기 때문에 가장 높은 점수를 부여합니다.

2. **신호 보내기 전략** (점수: 7점)
   - 구조 요

## 4. 결론
- LLM의 지능을 높이는 궁극적인 방법은, 모델 스스로 자신의 대답을 되돌아보게(Reflection) 하거나, 여러 경로를 탐색(ToT)하게 만드는 것입니다.
- 이 기법들을 12장의 LangGraph 제어문(Loop)과 결합하면, 최강의 자율형 에이전트가 탄생합니다.